# Phase 27 — Mixed-Benchmark Repair Validation (Phase 19)
## NeuroForge Experimental Research

Phase 18: the mixed construction erases the relational carrier on RC/FRC (ch0 add, then ch0:3 overwrite) while labels keep depending on it. Question: can the benchmark be repaired minimally so every label-generating component stays observable, preserving all other semantics? Repair first, models later — no architecture in this phase.

## 1. Phase 18 discovery

In [1]:
import json
p19 = json.load(open('../results/metrics/phase19_benchmark_validation/summary.json', encoding='utf-8'))
b = p19['aggregates']['bug']
print(f"original R-align: R={b['orig_R']*100:.1f}% RC={b['orig_RC']*100:.1f}% FRC={b['orig_FRC']*100:.1f}%")
print(f"erasure reproduced: {b['erasure_reproduced']}")

original R-align: R=97.8% RC=48.1% FRC=55.6%
erasure reproduced: True


## 2. Reproduce historical construction bug (19A)

In [2]:
import csv
for row in csv.DictReader(open('../results/metrics/phase19_benchmark_validation/bug_reproduction.csv')):
    if row['seed'] == str(p19['per_seed_results'][0]['seed']):
        print(f"{row['metric']}: {float(row['value'])*100:.1f}%" if float(row['value']) <= 1 else f"{row['metric']}: {row['value']}")

orig_R: 98.3%
orig_RC: 40.8%
orig_FRC: 51.7%
rep_R_ch5: 97.5%
erasure_reproduced: 100.0%


## 3. Exact overwrite dependency (19D ledger + empirical rates)

In [3]:
import csv
seen = 0
for row in csv.DictReader(open('../results/metrics/phase19_benchmark_validation/dependency_audit.csv')):
    if row['seed'] == str(p19['per_seed_results'][0]['seed']) and row['family'] in ('RC','FRC'):
        print(f"{row['construction']:>9} {row['family']} {row['component']}: ch={row['carrier_channel']} rate={float(row['observable_rate'])*100:.1f}% :: {row['later_transformation'][:70]}")
        seen += 1
        if seen >= 8:
            break

 original RC R: ch=0 rate=40.8% :: contextual block overwrites ch0:3 AFTER relational add (destroys R car
 original RC C: ch=0:3 rate=100.0% :: feature common-mode offsets (repaired) / none (original)
 original FRC R: ch=0 rate=51.7% :: contextual block overwrites ch0:3 AFTER relational add (destroys R car
 original FRC F: ch=0 rate=15.5% :: contextual block overwrites ch0:3 AFTER relational add (destroys R car
 original FRC C: ch=0:3 rate=100.0% :: feature common-mode offsets (repaired) / none (original)
 repaired RC R: ch=5 rate=96.7% :: no later write touches ch5 (carrier provably survives)
 repaired RC C: ch=0:3 rate=100.0% :: feature common-mode offsets (repaired) / none (original)
 repaired FRC R: ch=5 rate=98.3% :: no later write touches ch5 (carrier provably survives)


## 4. Minimal repair (versioned module; same RNG values, same label algebra)

In [4]:
import csv
for row in csv.DictReader(open('../results/metrics/phase19_benchmark_validation/construction_diff.csv')):
    print(f"{row['area']}: {row['before']} -> {row['after']}")
print('Repair: C first, F offsets after (common-mode), R carrier on ch5 (untouched).')

placement order: baseâ†’Fâ†’Râ†’C -> baseâ†’Câ†’Fâ†’R(ch5)
R carrier channel: ch0 (shared, overwritten) -> ch5 (previously pure noise; untouched by C/F writes)
F offsets: ch0/ch1 before C (erased on FC/FRC) -> ch0/ch1 after C (common-mode; key identity preserved)
C block: last -> first (identical writes)
labels: majority+parity -> majority+parity (identical algebra)
families/balance/sequences: 7 families, same counts, S=12 -> unchanged
Repair: C first, F offsets after (common-mode), R carrier on ch5 (untouched).


## 5. Observability audit (19C: R/F/C carriers per family, repaired)

In [5]:
import csv
for row in csv.DictReader(open('../results/metrics/phase19_benchmark_validation/observability.csv')):
    if row['seed'] == str(p19['per_seed_results'][0]['seed']):
        print(f"{row['metric']:>12}: {float(row['value'])*100:.1f}%")

   r_align_R: 97.5%
  r_align_RC: 96.7%
 r_align_FRC: 98.3%
  r_align_FR: 98.3%
     f_mag_F: 79.9%
    f_mag_FR: 79.9%
    f_mag_FC: 80.5%
   f_mag_FRC: 80.2%
     c_ret_C: 100.0%
    c_ret_RC: 100.0%
    c_ret_FC: 100.0%
   c_ret_FRC: 100.0%


## 6. Label-input dependency audit (19D: every label variable observable)

In [6]:
print('Reusable audit: statistic -> carrier channel -> later transforms -> rate.')
print('Ledger declares overwrites; empirical rates confirm. See dependency_audit.csv.')
o = p19['aggregates']['observability']
print(f"R-group: {[round(o[k]*100,1) for k in ('r_align_R','r_align_RC','r_align_FRC','r_align_FR')]}")

Reusable audit: statistic -> carrier channel -> later transforms -> rate.
Ledger declares overwrites; empirical rates confirm. See dependency_audit.csv.
R-group: [98.9, 97.2, 98.3, 98.6]


## 7. RC parity analysis (19E: genuine two-component requirement)

In [7]:
s = p19['aggregates']['semantics']
print(f"rule RC={s['rule_RC']*100:.1f}% vs C-only={s['c_only_RC']*100:.1f}% (margin bar 20pp)")
print(f"parity holds; FRC ties absent: {s['frc_no_tie']}; keys present: {s['keys_present']}")

rule RC=97.8% vs C-only=48.3% (margin bar 20pp)
parity holds; FRC ties absent: True; keys present: True


## 8. Counterfactual validation (19F: corrected methodology only)

In [8]:
import csv
for row in csv.DictReader(open('../results/metrics/phase19_benchmark_validation/counterfactual_validation.csv')):
    if row['seed'] == str(p19['per_seed_results'][0]['seed']):
        print(f"R-valid={float(row['R_valid_rate'])*100:.1f}% C-valid={float(row['C_valid_rate'])*100:.1f}% (bar 90%)")

R-valid=97.5% C-valid=96.7% (bar 90%)


## 9. Invariance controls (19G: C absolute, R comparative vs original)

In [9]:
import csv
for row in csv.DictReader(open('../results/metrics/phase19_benchmark_validation/invariance_controls.csv')):
    if row['seed'] == str(p19['per_seed_results'][0]['seed']):
        print(f"{row['metric']}: {float(row['value']):+.3f}")
print('Token moves shift positional carriers in BOTH versions (pre-existing); C retrieval must hold absolutely.')

c_ret_plain: +1.000
c_ret_perm: +1.000
c_retrieval_perm_drop: +0.000
r_align_plain_RC: +0.967
r_align_markvar_RC: +0.825
r_align_marker_delta: -0.142
r_align_marker_delta_original: +0.017
r_markvar_R_original: -0.183
r_markvar_R_repaired: -0.158
r_align_perm_repaired_drop: +0.483
r_align_perm_original_drop: +0.467
Token moves shift positional carriers in BOTH versions (pre-existing); C retrieval must hold absolutely.


## 10. Shortcut/leakage audit (19H: no direct copies, no new family leakage)

In [10]:
import csv
for row in csv.DictReader(open('../results/metrics/phase19_benchmark_validation/shortcut_audit.csv')):
    if row['seed'] == str(p19['per_seed_results'][0]['seed']):
        print(f"{row['metric']}: {float(row['value']):.3f}")

max_single_channel_label_probe: 0.767
family_probe_repaired: 0.171
family_probe_original: 0.136
family_probe_delta: 0.036


## 11. Shallow baselines (19I: rule + matched linear + tiny MLP context)

In [11]:
import csv
for row in csv.DictReader(open('../results/metrics/phase19_benchmark_validation/shallow_baselines.csv')):
    if row['seed'] == str(p19['per_seed_results'][0]['seed']):
        print(f"{row['metric']}: {float(row['value'])*100:.1f}%")
print('Raw-MLP limits on second-order stats are inductive-bias context, not validity failures.')

mlp_R: 46.7%
mlp_RC: 48.3%
mlp_FRC: 73.3%
stat_R: 97.5%
lagprobe_R: 89.5%
rule_RC: 96.7%
Raw-MLP limits on second-order stats are inductive-bias context, not validity failures.


## 12. Common-input contract (19K: all experts consume repaired batches)

In [12]:
import csv
for row in csv.DictReader(open('../results/metrics/phase19_benchmark_validation/common_contract.csv')):
    if row['seed'] == str(p19['per_seed_results'][0]['seed']):
        print(row)
print('Still [B,S,8]; marker semantics intact; no family/expert/label metadata.')

{'seed': '11', 'input_shape': '[32, 12, 8]', 'marker_present': 'True', 'finite': 'True'}
{'seed': '11', 'input_shape': 'mlp', 'marker_present': 'True', 'finite': '[32, 2]'}
{'seed': '11', 'input_shape': 'graph', 'marker_present': 'True', 'finite': '[32, 2]'}
{'seed': '11', 'input_shape': 'attention', 'marker_present': 'True', 'finite': '[32, 2]'}
{'seed': '11', 'input_shape': 'attention_v2', 'marker_present': 'True', 'finite': '[32, 2]'}
{'seed': '11', 'input_shape': 'joint', 'marker_present': 'True', 'finite': '[32, 2]'}
{'seed': '11', 'input_shape': 'joint_co', 'marker_present': 'True', 'finite': '[32, 2]'}
Still [B,S,8]; marker semantics intact; no family/expert/label metadata.


## 13. Validity gates G1-G8 (all must pass)

In [13]:
for g, d in p19['aggregates']['gates'].items():
    print(f"{g}: {'PASS' if d['passed'] else 'FAIL'} — {d['detail'][:110]}")

G1_observability: PASS — all component carriers observable
G2_no_leakage: PASS — max single-channel label probe 0.744 (bar 0.9)
G3_no_family_leakage: PASS — family-probe delta repaired−original -0.005 (tol 0.05)
G4_invariance: PASS — C retrieval perm drop +0.000; R-family markvar delta repaired -0.158 vs original -0.183 (RC-family deltas are 
G5_counterfactual: PASS — R-valid 0.969, C-valid 0.972 (bar 0.9)
G6_semantics: PASS — agree 61/120, disagree 59; FRC ties absent: True
G7_learnability: PASS — rule_RC 0.967; lagprobe_R 0.895; mlp R/RC/FRC [0.467, 0.483, 0.733] (context); stat_R 0.975
G8_reproducibility: PASS — exact tensor equality across rebuilds


## 14. H1-H8 (re-derived programmatically)

In [14]:
from neuroforge.evaluation.phase19_benchmark_validation import build_phase19_hypotheses
hyps = build_phase19_hypotheses(p19['aggregates'])
for h in sorted(hyps):
    print(f"{h}: {hyps[h]['status']}")
    print(f"    {hyps[h]['evidence']}")
assert all(hyps[h]['status'] == p19['hypotheses'][h]['status'] for h in hyps)
print('stored verdicts match fresh derivation: OK')

H1: SUPPORTED
    Original: R-align R 97.8% vs RC 48.1%/FRC 55.6%: erasure reproduced.
H2: SUPPORTED
    Repaired R-align on RC 97.2% (bar 90%).
H3: SUPPORTED
    Repaired R-align on FRC 98.3% (bar 90%).
H4: SUPPORTED
    All mixed-task components observable.
H5: SUPPORTED
    C retrieval perm drop +0.000; R-family markvar delta repaired -0.158 vs original -0.183 (RC-family deltas are floor effects, reported in CSV); perm R-drop repaired +0.483 vs original +0.467
H6: SUPPORTED
    max single-channel label probe 0.744; family delta -0.005.
H7: SUPPORTED
    Statistic+retrieval rule RC 97.8% vs C-only 48.3%: RC genuinely requires both.
H8: SUPPORTED
    All validity gates pass.
stored verdicts match fresh derivation: OK


## 15. Final CASE (programmatic)

In [15]:
from neuroforge.evaluation.phase19_benchmark_validation import select_phase19_case
case, label = select_phase19_case(hyps, p19['aggregates'])
print(f'Programmatic verdict: {case} — {label}')
assert case == p19['verdict_case']
print(f"Intervention recorded: {p19['minimal_intervention']['intervention']}")

Programmatic verdict: CASE F — Benchmark validity fully established and composition track can safely reopen
Intervention recorded: benchmark_repair


## 16. Recommendation on reopening composition experiments

In [16]:
from neuroforge.evaluation.phase19_benchmark_validation import recommendation_for_case
print('Recommendation:', recommendation_for_case(p19['verdict_case']))
print()
print('Loaded (not typed):')
print(f"  gates passed: {sum(1 for d in p19['aggregates']['gates'].values() if d['passed'])}/8")
print(f"  rule RC={p19['aggregates']['semantics']['rule_RC']*100:.1f}%, lagprobe R={p19['aggregates']['learnability']['lagprobe_R']*100:.1f}%")

Recommendation: Reopen composition with a clean portfolio re-evaluation first; no new architecture yet.

Loaded (not typed):
  gates passed: 8/8
  rule RC=97.8%, lagprobe R=90.7%
